<a href="https://colab.research.google.com/github/aparnapandey26/Stock-Market-Trend-Prediction-using-ML/blob/main/Stock_Market_Trend_Prediction_using_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

Download Data

In [4]:
data = yf.download("TCS.NS", period="1y")
print(data.head())

/tmp/ipykernel_2651/1675153222.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("TCS.NS", period="1y")
[*********************100%***********************]  1 of 1 completed

Price             Close         High          Low         Open   Volume
Ticker           TCS.NS       TCS.NS       TCS.NS       TCS.NS   TCS.NS
Date                                                                   
2025-04-23  3298.989746  3306.625755  3225.818529  3252.593172  3543923
2025-04-24  3287.970459  3298.506253  3274.534644  3287.487161  2252175
2025-04-25  3332.820557  3361.625143  3291.256959  3305.755889  2742991
2025-04-28  3328.470703  3342.293062  3291.256786  3320.254644  1593296
2025-04-29  3356.502197  3378.250592  3315.808441  3339.586779  1578332


Clean the data

In [5]:
data.dropna(inplace=True)

Feature Engineering

In [6]:
data["MA5"] = data["Close"].rolling(5).mean()
data["MA20"] = data["Close"].rolling(20).mean()

data["Return"] = data["Close"].pct_change()

data["Volatility"] = data["Return"].rolling(5).std()

Create Target Variable

In [7]:
data["Target"] = (data["Close"].shift(-1) > data["Close"]).astype(int)

Remove NaN Rows

In [9]:
data.dropna(inplace=True)

Select Features

In [10]:
X = data[["MA5", "MA20", "Return", "Volatility"]]
y = data["Target"]

Train Test Split

In [11]:
split = int(len(data) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

Train Model

In [12]:
model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

Prediction

In [13]:
pred = model.predict(X_test)

Accuracy Check

In [14]:
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.43478260869565216
              precision    recall  f1-score   support

           0       0.52      0.56      0.54        27
           1       0.29      0.26      0.28        19

    accuracy                           0.43        46
   macro avg       0.41      0.41      0.41        46
weighted avg       0.43      0.43      0.43        46



Predict Tomorrow

In [15]:
latest = X.tail(1)

tomorrow = model.predict(latest)

if tomorrow[0] == 1:
    print("Tomorrow: UP 📈")
else:
    print("Tomorrow: DOWN 📉")

Tomorrow: UP 📈


Save Model

In [16]:
import joblib

joblib.dump(model, "stock_model.pkl")

['stock_model.pkl']

Latest / More Data Load

In [46]:
data = yf.download("TCS.NS", period="5y")
data.dropna(inplace=True)
data.head()

/tmp/ipykernel_2651/1233541835.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("TCS.NS", period="5y")
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,TCS.NS,TCS.NS,TCS.NS,TCS.NS,TCS.NS
Date,,,,,
2021-04-23,2733.489990,2741.841222,2717.271062,2729.534144,1615309
2021-04-26,2725.841797,2771.729570,2714.633566,2726.940643,2490260
2021-04-27,2753.269043,2756.873344,2727.775811,2730.413042,1471417
2021-04-28,2746.324219,2769.971301,2735.687302,2769.048228,1639037
2021-04-29,2738.544678,2775.553906,2729.973677,2765.224751,1621395


New Features


In [47]:
data["MA5"] = data["Close"].rolling(5).mean()
data["MA10"] = data["Close"].rolling(10).mean()
data["MA20"] = data["Close"].rolling(20).mean()
data["MA50"] = data["Close"].rolling(50).mean()

data["Return"] = data["Close"].pct_change()
data["Volatility"] = data["Return"].rolling(5).std()

data["High_Low"] = data["High"] - data["Low"]
data["Open_Close"] = data["Open"] - data["Close"]

data.dropna(inplace=True)

Target

In [48]:
data["Target"] = (data["Close"].shift(-1) > data["Close"]).astype(int)
data.dropna(inplace=True)

Train Again

In [49]:
X = data[["MA5","MA10","MA20","MA50","Return","Volatility","High_Low","Open_Close"]]
y = data["Target"]

Train test model

In [50]:
split = int(len(data) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

Model Train (Random Forest Better Version)

In [61]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=10, n_estimators=200, random_state=42)

Accuracy Check

In [62]:
from sklearn.metrics import accuracy_score, classification_report

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.48739495798319327
              precision    recall  f1-score   support

           0       0.56      0.41      0.47       133
           1       0.44      0.58      0.50       105

    accuracy                           0.49       238
   macro avg       0.50      0.50      0.49       238
weighted avg       0.50      0.49      0.49       238



Use XGBoost

In [63]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [54]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [68]:
pred = model.predict(X_test)

In [70]:
acc = accuracy_score(y_test, pred)

print("Accuracy:", acc)
print(classification_report(y_test, pred))

Accuracy: 0.5126050420168067
              precision    recall  f1-score   support

           0       0.59      0.43      0.50       133
           1       0.46      0.62      0.53       105

    accuracy                           0.51       238
   macro avg       0.52      0.52      0.51       238
weighted avg       0.53      0.51      0.51       238

